In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin, clone

In [2]:
import sys
import os

sys.path.insert(0, os.path.abspath("../.."))
import nova.preprocessing as dp
from nova.preprocessing import train_val_test_split

In [3]:
from nova.layers import Dense
from nova.activations import Sigmoid, ReLu, Tanh, Softmax
from nova.losses import BinaryCrossEntropy, CategoricalCrossEntropy

In [4]:
df = pd.read_csv("C:\\Users\\Luca\\Desktop\\Personal\\Nova\\Examples\\v2\\dataset_voluntariado.csv")

In [5]:
data_preparer = dp.DataFramePreparer()

In [6]:
from nova.models import Sequential

In [7]:
model = Sequential()

In [8]:
from sklearn.metrics import f1_score
import numpy as np

In [9]:
train_set, val_set, test_set = train_val_test_split(df, stratify=["volunteer_type"])

X = df.drop(["volunteer_type", "ID"], axis=1)
X_prep = data_preparer.fit_transform(X)

In [10]:
model.add(Dense(input_dim=X_prep.shape[1], output_dim=32, initializer="he"))
model.add(ReLu())
model.add(Dense(input_dim=32, output_dim=16, initializer="he"))
model.add(ReLu())
model.add(Dense(input_dim=16, output_dim=8, initializer="he"))
model.add(Softmax())

In [11]:
model.compile(CategoricalCrossEntropy())

In [13]:
X_train = train_set.drop(["volunteer_type", "ID"], axis=1)
y_train = train_set["volunteer_type"]
X_test = test_set.drop(["volunteer_type", "ID"], axis=1)
y_test = test_set["volunteer_type"]
X_val = val_set.drop(["volunteer_type", "ID"], axis=1)
y_val = val_set["volunteer_type"]

X_train_prep = data_preparer.transform(X_train)
X_test_prep = data_preparer.transform(X_test)
X_val_prep = data_preparer.transform(X_val)

y_onehot_train = pd.get_dummies(y_train)
class_labels = y_onehot_train.columns
y_onehot_train = y_onehot_train.values.astype(float)

model.fit(X_train_prep, y_onehot_train, iterations_num=10000, optimization_algorithm="Adam", learning_rate=0.001)

train_predict = model.predict(X_train_prep)
train_predicted_indices = np.argmax(train_predict, axis=1)
train_predicted_labels = class_labels[train_predicted_indices]

test_predict = model.predict(X_test_prep)
test_predicted_indices = np.argmax(test_predict, axis=1)
test_predicted_labels = class_labels[test_predicted_indices]

val_predict = model.predict(X_val_prep)
val_predicted_indices = np.argmax(val_predict, axis=1)
val_predicted_labels = class_labels[val_predicted_indices]


print(f"Train score: {f1_score(y_train, train_predicted_labels, average='weighted')}")
print(f"Test score: {f1_score(y_test, test_predicted_labels, average='weighted')}")
print(f"Val score: {f1_score(y_val, val_predicted_labels, average='weighted')}")

0 0.16591511002829754
1000 0.15214745486917414
2000 0.14391296633187067
3000 0.13833263166923876
4000 0.1342835387037571
5000 0.1311908864799333
6000 0.12875234789193393
7000 0.1267652229725511
8000 0.12513592139213148
9000 0.1237687907450678
Train score: 0.9390085690333032
Test score: 0.9350855272825623
Val score: 0.9370426560408254
